In [ ]:
# imports

from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict , Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import BaseMessage, HumanMessage
import operator
import os 


In [ ]:
# loading 

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

In [ ]:
# define llm


# llm 
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0,
    google_api_key=GOOGLE_API_KEY
)


Chat model-এ যে messageগুলো পাঠানো হয়, যেমন:

HumanMessage
AIMessage
SystemMessage
ToolMessage

এসবই BaseMessage থেকে inherit করে।

LLM-এর conversation history একসাথে store করার জন্য।

In [ ]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):

    messages: Annotated[list[BaseMessage], add_messages]


In [ ]:
from langchain_core.messages import AIMessage

def chat_node(state):
    messages = state["messages"]
    response = llm.invoke(messages)

    return {
        "messages": [
            AIMessage(content=response[0]["text"])
        ]
    }

In [ ]:
checkpoint = MemorySaver()  # for persistance inside the ram

graph = StateGraph(ChatState)

# add nodes
graph.add_node('chat_node', chat_node)

#add edges
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)


chatbot = graph.compile(checkpointer=checkpoint) # store in ram 

In [ ]:
chatbot

In [ ]:
thread_id = "2"
initial_state = {
    'messages': [HumanMessage(content='What is my name?')]
}

config = {'configurable': {'thread_id': thread_id}}
response = chatbot.invoke(initial_state, config=config)

In [ ]:
chatbot.get_state(config=config)

In [ ]:
thread_id = "1" # unique id for specific user differnt differt user 

while True:
    user_message = input('Type here: ')

    print('User:', user_message)

    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        break

    config = {'configurable': {'thread_id': thread_id}}
    response = chatbot.invoke({'messages': [HumanMessage(content=user_message)]}, config=config)
    print('AI:', response['messages'][-1].content)

In [ ]:
thread_id = "3" # unique id for specific user differnt differt user 

while True:
    user_message = input('Type here: ')

    print('User:', user_message)

    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        break

    config = {'configurable': {'thread_id': thread_id}}
    response = chatbot.invoke({'messages': [HumanMessage(content=user_message)]}, config=config)
    print('AI:', response['messages'][-1].content)

In [ ]:
response['messages'][-1].content

In [ ]:
msg = response["messages"][-1]

if isinstance(msg, list):
    text = msg[0]["text"]
elif isinstance(msg, dict):
    text = msg["text"]
else:
    text = msg.content

print(text)